In [ ]:
%%capture
%pip install ultralytics roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="API-Key")
project = rf.workspace("ufvmestrado").project("abelhas-v3")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Extrair modelos para o colab

In [ ]:
import os

# 1. Definir os caminhos de origem
modelos_yolo_26 = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2/modelo_V2_09_02_2026/exportados_rpi5.zip"
modelos_yolo_11 = "/content/drive/MyDrive/Lesc-Bee videos/modelo_yolo_11/yolo_11_25_02_2026/exportados_rpi5.zip"
modelos_rtdetr = "/content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert/rt_dert_28_02_20262/exportados_rpi5.zip"

# 2. Criar os diretórios de destino
os.makedirs("/content/Modelos/yolo_26", exist_ok=True)
os.makedirs("/content/Modelos/yolo_11", exist_ok=True)
os.makedirs("/content/Modelos/rtdetr", exist_ok=True)

# 3. Extrair os arquivos (o parâmetro -q silencia a saída, -d define o destino)
print("Extraindo YOLO 26...")
!unzip -q "{modelos_yolo_26}" -d "/content/Modelos/yolo_26"

print("Extraindo YOLO 11...")
!unzip -q "{modelos_yolo_11}" -d "/content/Modelos/yolo_11"

print("Extraindo RT-DETR...")
!unzip -q "{modelos_rtdetr}" -d "/content/Modelos/rtdetr"

print("Concluído!")

## Fase 1: Avaliação em GPU

In [ ]:
import pandas as pd
from ultralytics import YOLO, RTDETR
import os
import torch
import gc
import numpy as np

# --- Configurações Iniciais ---
dataset_yaml = "/content/Abelhas-V3-1/data.yaml"
csv_saida = "/content/drive/MyDrive/Lesc-Bee videos/resultados_modelos_gpu.csv"

# EXCLUSÃO EXPLÍCITA DO ARQUIVO ANTIGO PARA EVITAR DUPLICAÇÃO DE DADOS
if os.path.exists(csv_saida):
    os.remove(csv_saida)
    print(f"🗑️ Arquivo antigo removido: {csv_saida}")

caminho_base_yolo11 = "/content/Modelos/yolo_11"
caminho_base_yolo26 = "/content/Modelos/yolo_26"
caminho_base_rtdetr = "/content/Modelos/rtdetr"

modelos_para_testar = [
    {"nome": "YOLO11n", "formato": "ONNX_FP32", "path": f"{caminho_base_yolo11}/modelo_onnx_fp32.onnx", "tipo": "YOLO"},
    {"nome": "YOLO11n", "formato": "ONNX_FP16", "path": f"{caminho_base_yolo11}/modelo_onnx_fp16.onnx", "tipo": "YOLO"},
    {"nome": "YOLO26n", "formato": "ONNX_FP32", "path": f"{caminho_base_yolo26}/modelo_onnx_fp32.onnx", "tipo": "YOLO"},
    {"nome": "YOLO26n", "formato": "ONNX_FP16", "path": f"{caminho_base_yolo26}/modelo_onnx_fp16.onnx", "tipo": "YOLO"},
    {"nome": "RT-DETR", "formato": "ONNX_FP32", "path": f"{caminho_base_rtdetr}/modelo_onnx_fp32.onnx", "tipo": "RTDETR"},
    {"nome": "RT-DETR", "formato": "ONNX_FP16", "path": f"{caminho_base_rtdetr}/modelo_onnx_fp16.onnx", "tipo": "RTDETR"}
]

resultados = []

print("\n🚀 Iniciando Fase 1: Testes de Inferência na GPU...\n")

for config in modelos_para_testar:
    print(f"--- Avaliando {config['nome']} ({config['formato']}) ---")

    if not os.path.exists(config['path']):
        print(f"⚠️ Arquivo não encontrado: {config['path']} - Pulando para o próximo...\n")
        continue

    try:
        if config['tipo'] == "YOLO":
            model = YOLO(config['path'], task='detect')
        else:
            model = RTDETR(config['path'])

        metrics = model.val(data=dataset_yaml, device='0', imgsz=640, split='val', verbose=False, workers=0, batch=1)

        # === MÉTRICAS GLOBAIS ===
        map50 = metrics.box.map50
        map75 = metrics.box.map75
        map50_95 = metrics.box.map

        latencia_ms = sum(metrics.speed.values())
        fps = 1000.0 / latencia_ms if latencia_ms > 0 else 0

        # === MÉTRICAS POR CLASSE ===
        class_indices = metrics.box.ap_class_index
        precisions = metrics.box.p
        recalls = metrics.box.r

        # Arrays do Ultralytics para mAP por classe
        maps50 = metrics.box.ap50 # mAP@0.5 para cada classe
        maps50_95 = metrics.box.ap # mAP@0.5:0.95 para cada classe

        # Para extrair o mAP75 por classe, acessamos a matriz de todos os APs
        # A matriz tem formato (N_classes, 10 thresholds). O threshold 0.75 é o índice 5.
        try:
            # Acessando o atributo oculto onde a Ultralytics guarda as métricas completas
            todos_aps = metrics.box.all_ap
            maps75 = todos_aps[:, 5]
        except Exception as e:
            # Fallback caso a versão do Ultralytics não tenha essa matriz exata
            print("  [Aviso] Não foi possível extrair mAP75_classe. Preenchendo com nulo.")
            maps75 = [np.nan] * len(class_indices)

        # Populando o dicionário da linha
        linha_resultado = {
            "Modelo": config['nome'],
            "Formato": config['formato'],
            "Dispositivo": "GPU",
            "mAP50_Global": round(map50, 4),
            "mAP75_Global": round(map75, 4),
            "mAP50-95_Global": round(map50_95, 4),
            "Latencia_ms_GPU": round(latencia_ms, 2),
            "FPS_GPU": round(fps, 2)
        }

        # Adicionando as métricas dinâmicas para cada classe
        for idx, c_idx in enumerate(class_indices):
            nome_classe = metrics.names[c_idx]
            linha_resultado[f"Precision_{nome_classe}"] = round(precisions[idx], 4)
            linha_resultado[f"Recall_{nome_classe}"] = round(recalls[idx], 4)

            linha_resultado[f"mAP50_{nome_classe}"] = round(maps50[idx], 4)
            linha_resultado[f"mAP75_{nome_classe}"] = round(maps75[idx], 4) if not np.isnan(maps75[idx]) else None
            linha_resultado[f"mAP50-95_{nome_classe}"] = round(maps50_95[idx], 4)

        resultados.append(linha_resultado)
        print(f"✅ Sucesso! FPS obtido (End-to-End): {linha_resultado['FPS_GPU']}\n")

    except Exception as e:
        print(f"❌ Erro ao avaliar {config['nome']} ({config['formato']}): {e}\n")

    finally:
        if 'model' in locals(): del model
        if 'metrics' in locals(): del metrics
        torch.cuda.empty_cache()
        gc.collect()

if resultados:
    df_gpu = pd.DataFrame(resultados)

    # Definir a ordem desejada das colunas no CSV para ficar organizado
    colunas_ordenadas = [
        "Modelo", "Formato", "Dispositivo", "Latencia_ms_GPU", "FPS_GPU",
        "mAP50_Global", "mAP75_Global", "mAP50-95_Global",
        "Precision_bee", "Recall_bee", "mAP50_bee", "mAP75_bee", "mAP50-95_bee",
        "Precision_pollen", "Recall_pollen", "mAP50_pollen", "mAP75_pollen", "mAP50-95_pollen"
    ]
    # Filtrar apenas as colunas que realmente existem no df (para o caso do nome das classes ser diferente)
    colunas_ordenadas = [col for col in colunas_ordenadas if col in df_gpu.columns]

    # Adicionar colunas extras que não estavam na lista de ordenação (caso existam mais classes no futuro)
    outras_colunas = [col for col in df_gpu.columns if col not in colunas_ordenadas]
    colunas_finais = colunas_ordenadas + outras_colunas

    df_gpu = df_gpu[colunas_finais]
    df_gpu.to_csv(csv_saida, index=False)
    print(f"🎉 Fase 1 Concluída! CSV gerado com sucesso em:\n{csv_saida}")
else:
    print("Nenhum resultado foi processado. Verifique os caminhos dos modelos.")

## Fase 2: Avaliação em CPU

In [ ]:
import pandas as pd
from ultralytics import YOLO, RTDETR
import os
import gc
import numpy as np

# --- Configurações Iniciais ---
dataset_yaml = "/content/Abelhas-V3-1/data.yaml"
csv_resultados = "/content/drive/MyDrive/Lesc-Bee videos/resultados_modelos_cpu.csv"

# EXCLUSÃO EXPLÍCITA DO ARQUIVO ANTIGO PARA EVITAR DUPLICAÇÃO DE DADOS
if os.path.exists(csv_resultados):
    os.remove(csv_resultados)
    print(f"🗑️ Arquivo antigo removido: {csv_resultados}")

caminho_base_yolo11 = "/content/Modelos/yolo_11"
caminho_base_yolo26 = "/content/Modelos/yolo_26"
caminho_base_rtdetr = "/content/Modelos/rtdetr"

modelos_cpu = [
    {"nome": "YOLO11n", "formato": "ONNX_FP32", "path": f"{caminho_base_yolo11}/modelo_onnx_fp32.onnx", "tipo": "YOLO"},
    {"nome": "YOLO11n", "formato": "ONNX_FP16", "path": f"{caminho_base_yolo11}/modelo_onnx_fp16.onnx", "tipo": "YOLO"},
    {"nome": "YOLO11n", "formato": "OpenVINO_INT8", "path": f"{caminho_base_yolo11}/int8_openvino_model", "tipo": "YOLO"},

    {"nome": "YOLO26n", "formato": "ONNX_FP32", "path": f"{caminho_base_yolo26}/modelo_onnx_fp32.onnx", "tipo": "YOLO"},
    {"nome": "YOLO26n", "formato": "ONNX_FP16", "path": f"{caminho_base_yolo26}/modelo_onnx_fp16.onnx", "tipo": "YOLO"},
    {"nome": "YOLO26n", "formato": "OpenVINO_INT8", "path": f"{caminho_base_yolo26}/int8_openvino_model", "tipo": "YOLO"},

    {"nome": "RT-DETR", "formato": "ONNX_FP32", "path": f"{caminho_base_rtdetr}/modelo_onnx_fp32.onnx", "tipo": "RTDETR"},
    {"nome": "RT-DETR", "formato": "ONNX_FP16", "path": f"{caminho_base_rtdetr}/modelo_onnx_fp16.onnx", "tipo": "RTDETR"},
    {"nome": "RT-DETR", "formato": "OpenVINO_INT8", "path": f"{caminho_base_rtdetr}/int8_openvino_model", "tipo": "RTDETR"}
]

resultados_cpu = []

print("\n🚀 Iniciando Fase 2: Testes de Inferência na CPU...\n")

for config in modelos_cpu:
    print(f"--- Avaliando {config['nome']} ({config['formato']}) na CPU ---")

    if not os.path.exists(config['path']):
        print(f"⚠️ Ficheiro/Diretório não encontrado: {config['path']} - A saltar...\n")
        continue

    try:
        if config['tipo'] == "YOLO":
            model = YOLO(config['path'], task='detect')
        else:
            model = RTDETR(config['path'])

        metrics = model.val(data=dataset_yaml, device='cpu', imgsz=640, split='val', verbose=False, workers=0, batch=1)

        # === MÉTRICAS GLOBAIS ===
        map50 = metrics.box.map50
        map75 = metrics.box.map75
        map50_95 = metrics.box.map

        # NOTA METODOLÓGICA PARA O ARTIGO:
        # Latência End-to-End calculada somando Pre-process + Inference + Post-process (NMS)
        latencia_ms = sum(metrics.speed.values())
        fps = 1000.0 / latencia_ms if latencia_ms > 0 else 0

        # === MÉTRICAS POR CLASSE ===
        class_indices = metrics.box.ap_class_index
        precisions = metrics.box.p
        recalls = metrics.box.r
        maps50 = metrics.box.ap50
        maps50_95 = metrics.box.ap # mAP@0.5:0.95 para cada classe

        # Extração do mAP75 (índice 5 da matriz de APs)
        try:
            todos_aps = metrics.box.all_ap
            maps75 = todos_aps[:, 5]
        except Exception as e:
            print("  [Aviso] Não foi possível extrair mAP75_classe. Preenchendo com nulo.")
            maps75 = [np.nan] * len(class_indices)

        linha_resultado = {
            "Modelo": config['nome'],
            "Formato": config['formato'],
            "Dispositivo": "CPU",
            "Latencia_ms_CPU": round(latencia_ms, 2),
            "FPS_CPU": round(fps, 2),
            "mAP50_Global": round(map50, 4),
            "mAP75_Global": round(map75, 4),
            "mAP50-95_Global": round(map50_95, 4)
        }

        # Iterando pelas classes detectadas
        for idx, c_idx in enumerate(class_indices):
            nome_classe = metrics.names[c_idx]
            linha_resultado[f"Precision_{nome_classe}"] = round(precisions[idx], 4)
            linha_resultado[f"Recall_{nome_classe}"] = round(recalls[idx], 4)

            linha_resultado[f"mAP50_{nome_classe}"] = round(maps50[idx], 4)
            linha_resultado[f"mAP75_{nome_classe}"] = round(maps75[idx], 4) if not np.isnan(maps75[idx]) else None
            linha_resultado[f"mAP50-95_{nome_classe}"] = round(maps50_95[idx], 4)

        resultados_cpu.append(linha_resultado)
        print(f"✅ Sucesso! FPS obtido (End-to-End CPU): {linha_resultado['FPS_CPU']}\n")

    except Exception as e:
        print(f"❌ Erro ao avaliar {config['nome']} ({config['formato']}): {e}\n")

    finally:
        if 'model' in locals(): del model
        if 'metrics' in locals(): del metrics
        gc.collect()

if resultados_cpu:
    df_final = pd.DataFrame(resultados_cpu)

    # Garantir a mesma ordem visual e estrutural do CSV da GPU
    colunas_ordenadas = [
        "Modelo", "Formato", "Dispositivo", "Latencia_ms_CPU", "FPS_CPU",
        "mAP50_Global", "mAP75_Global", "mAP50-95_Global",
        "Precision_bee", "Recall_bee", "mAP50_bee", "mAP75_bee", "mAP50-95_bee",
        "Precision_pollen", "Recall_pollen", "mAP50_pollen", "mAP75_pollen", "mAP50-95_pollen"
    ]

    colunas_ordenadas = [col for col in colunas_ordenadas if col in df_final.columns]
    outras_colunas = [col for col in df_final.columns if col not in colunas_ordenadas]
    colunas_finais = colunas_ordenadas + outras_colunas

    df_final = df_final[colunas_finais]
    df_final.to_csv(csv_resultados, index=False)
    print(f"🎉 Fase 2 Concluída! CSV gerado com os testes de CPU em:\n{csv_resultados}")
else:
    print("Nenhum resultado processado na CPU. Verifique os caminhos dos modelos.")

## Fase 3 Teste de Tracking na GPU

In [ ]:
!pip install motmetrics
!pip install "numpy<2"

In [ ]:
# ================================================
# 1. Configuração do Rastreador ByteTrack
# ================================================
# track_buffer de 360 alinhado com vídeo a 120 FPS = 3 segundos de tolerância
yaml_content = """
tracker_type: bytetrack
track_high_thresh: 0.50
track_low_thresh: 0.10
new_track_thresh: 0.60
track_buffer: 360
match_thresh: 0.95
gmc_method: none
appearance_thresh: 0.25
with_reid: False
fuse_score: True
"""

tracker_config = "custom_bytetrack.yaml"
with open(tracker_config, "w") as f:
    f.write(yaml_content)

print(f"✅ Ficheiro {tracker_config} criado com sucesso e pronto para o artigo!")

import os
import pandas as pd
import numpy as np
import motmetrics as mm
import torch
import gc
from ultralytics import YOLO, RTDETR

# --- Configurações Iniciais ---
video_teste = "/content/trecho_extraido.mp4"
gt_path = "/content/MOTA/gt.txt"
csv_tracking_saida = "/content/resultados_tracking_gpu.csv"

caminho_base_yolo11 = "/content/Modelos/yolo_11/modelo_onnx_fp32.onnx"
caminho_base_yolo26 = "/content/Modelos/yolo_26/modelo_onnx_fp32.onnx"
caminho_base_rtdetr = "/content/Modelos/rtdetr/modelo_onnx_fp32.onnx"

modelos_tracking = [
    {"nome": "YOLO11n + ByteTrack", "path": caminho_base_yolo11, "tipo": "YOLO"},
    {"nome": "YOLO26n + ByteTrack", "path": caminho_base_yolo26, "tipo": "YOLO"},
    {"nome": "RT-DETR + ByteTrack", "path": caminho_base_rtdetr, "tipo": "RTDETR"}
]

# --- 1. Carregar Ground Truth ---
print("📥 Carregando Ground Truth...")
df_gt = pd.read_csv(gt_path, header=None, names=['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'class', 'vis'])

# BLINDAGEM DO FRAME INICIAL: Identifica se o GT começa no frame 0 ou 1
primeiro_frame_gt = int(df_gt['frame'].min())
print(f"🛡️ BLINDAGEM MOT: O Ground Truth inicia no frame {primeiro_frame_gt}. O loop de avaliação será alinhado a este valor.")

resultados_tracking = []

print("\n🚀 Iniciando Fase 3: Avaliação de Tracking (MOTA) na GPU...\n")

for config in modelos_tracking:
    print(f"--- Avaliando {config['nome']} ---")

    if not os.path.exists(config['path']):
        print(f"⚠️ Arquivo não encontrado: {config['path']} - Pulando...\n")
        continue

    try:
        if config['tipo'] == "YOLO":
            model = YOLO(config['path'], task='detect')
        else:
            model = RTDETR(config['path'])

        results = model.track(source=video_teste, device='0', persist=True, tracker="custom_bytetrack.yaml", stream=True, verbose=False)

        acc = mm.MOTAccumulator(auto_id=True)

        for i, result in enumerate(results):
            # Alinha o frame atual predito com o Ground Truth dinamicamente
            frame_idx = primeiro_frame_gt + i

            pred_boxes = []
            pred_ids = []

            if result.boxes is not None and result.boxes.id is not None:
                boxes_xywh = result.boxes.xywh.cpu().numpy()
                ids = result.boxes.id.int().cpu().tolist()

                for box, track_id in zip(boxes_xywh, ids):
                    cx, cy, w, h = box
                    tl_x = cx - (w / 2)
                    tl_y = cy - (h / 2)
                    pred_boxes.append([tl_x, tl_y, w, h])
                    pred_ids.append(track_id)

            gt_frame = df_gt[df_gt['frame'] == frame_idx]
            gt_ids = gt_frame['id'].tolist()
            gt_boxes = gt_frame[['bb_left', 'bb_top', 'bb_width', 'bb_height']].values.tolist()

            distance_matrix = mm.distances.iou_matrix(gt_boxes, pred_boxes, max_iou=0.5)
            acc.update(gt_ids, pred_ids, distance_matrix)

        print("📊 Computando métricas MOT...")
        mh = mm.metrics.create()
        summary = mh.compute(acc, metrics=['mota', 'idf1', 'num_false_positives', 'num_misses', 'num_switches'], name=config['nome'])

        if not summary.empty:
            mota = summary.loc[config['nome'], 'mota'] * 100
            idf1 = summary.loc[config['nome'], 'idf1'] * 100
            fp = summary.loc[config['nome'], 'num_false_positives']
            fn = summary.loc[config['nome'], 'num_misses']
            idsw = summary.loc[config['nome'], 'num_switches']

            linha = {
                "Modelo": config['nome'],
                "MOTA (%)": round(mota, 2) if not pd.isna(mota) else 0.0,
                "IDF1 (%)": round(idf1, 2) if not pd.isna(idf1) else 0.0,
                "Falsos Positivos (FP)": int(fp),
                "Falsos Negativos (FN)": int(fn),
                "ID Switches (IDSW)": int(idsw)
            }

            resultados_tracking.append(linha)
            print(f"✅ Concluído: MOTA: {linha['MOTA (%)']}% | IDF1: {linha['IDF1 (%)']}% | IDSW: {linha['ID Switches (IDSW)']}\n")
        else:
            print("⚠️ Nenhuma métrica gerada. O modelo falhou em prever.")

    except Exception as e:
        print(f"❌ Erro ao avaliar {config['nome']}: {e}\n")

    finally:
        if 'model' in locals(): del model
        if 'results' in locals(): del results
        if 'acc' in locals(): del acc
        torch.cuda.empty_cache()
        gc.collect()

if resultados_tracking:
    df_tracking = pd.DataFrame(resultados_tracking)
    df_tracking.to_csv(csv_tracking_saida, index=False)
    print(f"🎉 Fase 3 Concluída! CSV de Tracking gerado com sucesso em:\n{csv_tracking_saida}")

Contagem Direcional via Python

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

def extrair_e_visualizar_roi(video_path, roi=None):
    """
    Extrai o primeiro frame de um vídeo. Se as coordenadas da ROI forem fornecidas,
    desenha o retângulo de validação antes de exibir e salvar.
    """
    # 1. Abrir o arquivo de vídeo
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"❌ Erro: Não foi possível abrir o vídeo no caminho: {video_path}")
        return

    # 2. Ler apenas o primeiro frame
    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("❌ Erro: Não foi possível capturar o primeiro frame do vídeo.")
        return

    # O OpenCV carrega as imagens no padrão BGR. Vamos converter para RGB para exibição correta.
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 3. Lógica Condicional: Com ROI vs Sem ROI
    if roi is not None:
        xmin, ymin = roi['xmin'], roi['ymin']
        xmax, ymax = roi['xmax'], roi['ymax']

        # Desenhar o retângulo delimitador (Cor vermelha em RGB: 255, 0, 0 | Espessura: 3)
        cv2.rectangle(frame_rgb, (xmin, ymin), (xmax, ymax), (255, 0, 0), 3)
        output_path = "primeiro_frame_com_roi_validada.jpg"
        titulo = "Validação: Região de Interesse (ROI) Marcada"
        print(f"✅ ROI recebida. Desenhando marcação nas coordenadas: {roi}")
    else:
        output_path = "primeiro_frame_limpo.jpg"
        titulo = "Modo de Coleta: Imagem original para extração de coordenadas"
        print("📸 Nenhuma coordenada fornecida (roi=None). Extraindo apenas a imagem limpa.")

    # 4. Salvar a imagem resultante no disco (convertendo de volta para BGR)
    cv2.imwrite(output_path, cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
    print(f"💾 Imagem salva com sucesso com o nome: {output_path}")

    # 5. Renderizar a imagem diretamente no output do notebook
    plt.figure(figsize=(12, 8))
    plt.imshow(frame_rgb)
    plt.title(titulo)
    plt.axis("off") # Oculta os eixos para uma visualização mais limpa
    plt.show()

# =====================================================================
# ÁREA DE EXECUÇÃO
# =====================================================================
caminho_do_video = "/content/trecho_extraido.mp4"

# PASSO 1: Rode primeiro com roi=None para extrair a imagem limpa.
# Faça o download do arquivo "primeiro_frame_limpo.jpg" gerado, abra em um
# editor de imagens (Paint, Preview, Photopea) e anote os 4 pontos.
#extrair_e_visualizar_roi(caminho_do_video, roi=None)

print("-" * 50)

# PASSO 2: Após descobrir os pixels, descomente as linhas abaixo,
# insira os seus valores reais e rode o código novamente para conferir o alinhamento.

ROI_TESTE = {"xmin": 415, "ymin": 415, "xmax": 540, "ymax": 580}
extrair_e_visualizar_roi(caminho_do_video, roi=ROI_TESTE)

In [ ]:
import os
import pandas as pd
import numpy as np
import motmetrics as mm
import torch
import gc
from ultralytics import YOLO, RTDETR

# ==========================================
# DEFINA AQUI AS COORDENADAS DA SUA ROI
# (xmin, ymin, xmax, ymax) em pixels
# ==========================================
ROI = {"xmin": 415, "ymin": 415, "xmax": 540, "ymax": 580}

def ponto_dentro_roi(x, y, roi):
    """Verifica se a coordenada (x,y) está dentro do retângulo da ROI"""
    return (roi["xmin"] <= x <= roi["xmax"]) and (roi["ymin"] <= y <= roi["ymax"])

def classificar_fluxo(trajetorias, roi):
    """Aplica a lógica da Seção 3.3 para contar Entradas, Saídas e Voos Incertos"""
    contagem = {"Entrada": 0, "Saida": 0, "Incertos": 0}

    for track_id, dados in trajetorias.items():
        inicio_x, inicio_y = dados["primeiro_ponto"]
        fim_x, fim_y = dados["ultimo_ponto"]

        inicio_dentro = ponto_dentro_roi(inicio_x, inicio_y, roi)
        fim_dentro = ponto_dentro_roi(fim_x, fim_y, roi)

        if not inicio_dentro and fim_dentro:
            contagem["Entrada"] += 1
        elif inicio_dentro and not fim_dentro:
            contagem["Saida"] += 1
        else:
            contagem["Incertos"] += 1

    return contagem

# --- Configurações Iniciais ---
video_teste = "/content/trecho_extraido.mp4"
gt_path = "/content/MOTA/gt.txt"
csv_tracking_saida = "/content/resultados_tracking_gpu.csv"
csv_contagem_saida = "/content/resultados_contagem_direcional.csv"

caminho_base_yolo11 = "/content/Modelos/yolo_11/modelo_onnx_fp32.onnx"
caminho_base_yolo26 = "/content/Modelos/yolo_26/modelo_onnx_fp32.onnx"
caminho_base_rtdetr = "/content/Modelos/rtdetr/modelo_onnx_fp32.onnx"

modelos_tracking = [
    {"nome": "YOLO11n + ByteTrack", "path": caminho_base_yolo11, "tipo": "YOLO"},
    {"nome": "YOLO26n + ByteTrack", "path": caminho_base_yolo26, "tipo": "YOLO"},
    {"nome": "RT-DETR + ByteTrack", "path": caminho_base_rtdetr, "tipo": "RTDETR"}
]

# --- 1. Processar Ground Truth para Contagem ---
print("📥 Carregando e processando Ground Truth...")
df_gt = pd.read_csv(gt_path, header=None, names=['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'class', 'vis'])
primeiro_frame_gt = int(df_gt['frame'].min())

# Extrair trajetórias reais (Ground Truth)
traj_gt = {}
for _, row in df_gt.iterrows():
    track_id = int(row['id'])
    # Calcula o centro (cx, cy) da bounding box
    cx = row['bb_left'] + (row['bb_width'] / 2)
    cy = row['bb_top'] + (row['bb_height'] / 2)

    if track_id not in traj_gt:
        traj_gt[track_id] = {"primeiro_ponto": (cx, cy), "ultimo_ponto": (cx, cy)}
    else:
        traj_gt[track_id]["ultimo_ponto"] = (cx, cy)

contagem_gt = classificar_fluxo(traj_gt, ROI)
print(f"🐝 Realidade (Ground Truth): {contagem_gt['Entrada']} Entradas, {contagem_gt['Saida']} Saídas.")

resultados_tracking = []
resultados_contagem = []

print("\n🚀 Iniciando Avaliação de Tracking (MOTA) e Contagem na GPU...\n")

for config in modelos_tracking:
    print(f"--- Avaliando {config['nome']} ---")
    if not os.path.exists(config['path']):
        print(f"⚠️ Arquivo não encontrado: {config['path']} - Pulando...\n")
        continue

    traj_preditas = {} # Dicionário para armazenar a trajetória deste modelo

    try:
        model = YOLO(config['path'], task='detect') if config['tipo'] == "YOLO" else RTDETR(config['path'])
        results = model.track(source=video_teste, device='0', persist=True, tracker="custom_bytetrack.yaml", stream=True, verbose=False)
        acc = mm.MOTAccumulator(auto_id=True)

        for i, result in enumerate(results):
            frame_idx = primeiro_frame_gt + i
            pred_boxes = []
            pred_ids = []

            if result.boxes is not None and result.boxes.id is not None:
                boxes_xywh = result.boxes.xywh.cpu().numpy()
                ids = result.boxes.id.int().cpu().tolist()

                for box, track_id in zip(boxes_xywh, ids):
                    cx, cy, w, h = box
                    tl_x = cx - (w / 2)
                    tl_y = cy - (h / 2)

                    pred_boxes.append([tl_x, tl_y, w, h])
                    pred_ids.append(track_id)

                    # --- LÓGICA DE TRAJETÓRIA ADICIONADA AQUI ---
                    if track_id not in traj_preditas:
                        traj_preditas[track_id] = {"primeiro_ponto": (cx, cy), "ultimo_ponto": (cx, cy)}
                    else:
                        traj_preditas[track_id]["ultimo_ponto"] = (cx, cy)
                    # --------------------------------------------

            gt_frame = df_gt[df_gt['frame'] == frame_idx]
            gt_ids = gt_frame['id'].tolist()
            gt_boxes = gt_frame[['bb_left', 'bb_top', 'bb_width', 'bb_height']].values.tolist()

            distance_matrix = mm.distances.iou_matrix(gt_boxes, pred_boxes, max_iou=0.5)
            acc.update(gt_ids, pred_ids, distance_matrix)

        print("📊 Computando métricas MOT e Fluxo...")
        mh = mm.metrics.create()
        summary = mh.compute(acc, metrics=['mota', 'idf1', 'num_false_positives', 'num_misses', 'num_switches'], name=config['nome'])

        # Cálculo do fluxo usando a lógica da Seção 3.3
        contagem_modelo = classificar_fluxo(traj_preditas, ROI)

        # Erro Absoluto
        erro_entrada = abs(contagem_gt['Entrada'] - contagem_modelo['Entrada'])
        erro_saida = abs(contagem_gt['Saida'] - contagem_modelo['Saida'])

        if not summary.empty:
            mota = summary.loc[config['nome'], 'mota'] * 100
            idf1 = summary.loc[config['nome'], 'idf1'] * 100

            # Linha original do seu MOTA
            linha_mot = {
                "Modelo": config['nome'], "MOTA (%)": round(mota, 2) if not pd.isna(mota) else 0.0,
                "IDF1 (%)": round(idf1, 2) if not pd.isna(idf1) else 0.0,
                "FP": int(summary.loc[config['nome'], 'num_false_positives']),
                "FN": int(summary.loc[config['nome'], 'num_misses']),
                "IDSW": int(summary.loc[config['nome'], 'num_switches'])
            }
            resultados_tracking.append(linha_mot)

            # Nova linha para a tabela de contagem
            linha_contagem = {
                "Modelo": config['nome'],
                "GT Entrada": contagem_gt['Entrada'],
                "Pred Entrada": contagem_modelo['Entrada'],
                "Erro Absoluto (In)": erro_entrada,
                "GT Saída": contagem_gt['Saida'],
                "Pred Saída": contagem_modelo['Saida'],
                "Erro Absoluto (Out)": erro_saida
            }
            resultados_contagem.append(linha_contagem)

            print(f"✅ MOTA: {linha_mot['MOTA (%)']}% | IDF1: {linha_mot['IDF1 (%)']}%")
            print(f"🔄 Fluxo Predito -> Entradas: {contagem_modelo['Entrada']} (Erro: {erro_entrada}) | Saídas: {contagem_modelo['Saida']} (Erro: {erro_saida})\n")

    except Exception as e:
        print(f"❌ Erro ao avaliar {config['nome']}: {e}\n")

    finally:
        if 'model' in locals(): del model
        if 'results' in locals(): del results
        if 'acc' in locals(): del acc
        torch.cuda.empty_cache()
        gc.collect()

# Salvar Arquivos
if resultados_tracking:
    pd.DataFrame(resultados_tracking).to_csv(csv_tracking_saida, index=False)
    pd.DataFrame(resultados_contagem).to_csv(csv_contagem_saida, index=False)
    print(f"🎉 Avaliação Concluída! CSVs gerados:\n1. {csv_tracking_saida}\n2. {csv_contagem_saida}")

Salva um video mostrando o real e o estimado pelo modelo. (para visualização)

In [ ]:
import os
import cv2
import pandas as pd
import torch
import gc
from ultralytics import YOLO, RTDETR

yaml_content = """
tracker_type: bytetrack
track_high_thresh: 0.50
track_low_thresh: 0.10
new_track_thresh: 0.60
track_buffer: 360
match_thresh: 0.95
gmc_method: none
appearance_thresh: 0.25
with_reid: False
fuse_score: True
"""

tracker_config = "custom_bytetrack.yaml"
with open(tracker_config, "w") as f:
    f.write(yaml_content)

print(f"✅ Ficheiro {tracker_config} criado com sucesso e pronto para o artigo!")


# --- Configurações Iniciais ---
video_teste = "/content/video_validacao_mota_30fps.mp4"
gt_path = "/content/MOTA/gt.txt"
pasta_saida = "/content/"

os.makedirs(pasta_saida, exist_ok=True)

caminho_base_yolo11 = "/content/Modelos/yolo_11/modelo_onnx_fp32.onnx"
caminho_base_yolo26 = "/content/Modelos/yolo_26/modelo_onnx_fp32.onnx"
caminho_base_rtdetr = "/content/Modelos/rtdetr/modelo_onnx_fp32.onnx"

modelos_tracking = [
    {"nome": "YOLO11n + ByteTrack", "path": caminho_base_yolo11, "tipo": "YOLO"},
    {"nome": "YOLO26n + ByteTrack", "path": caminho_base_yolo26, "tipo": "YOLO"},
    {"nome": "RT-DETR + ByteTrack", "path": caminho_base_rtdetr, "tipo": "RTDETR"}
]

print("📥 Carregando Ground Truth para o desenho das caixas originais...")
df_gt = pd.read_csv(gt_path, header=None, names=['frame', 'id', 'bb_left', 'bb_top', 'bb_width', 'bb_height', 'conf', 'class', 'vis'])
primeiro_frame_gt = int(df_gt['frame'].min())

cap = cv2.VideoCapture(video_teste)
fps_video = cap.get(cv2.CAP_PROP_FPS)
largura_video = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura_video = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print("\n🚀 Iniciando Geração dos Vídeos de Comparação (Real vs. Modelo)...\n")

for config in modelos_tracking:
    print(f"--- Processando vídeo para: {config['nome']} ---")

    if not os.path.exists(config['path']):
        continue

    try:
        if config['tipo'] == "YOLO":
            model = YOLO(config['path'], task='detect')
        else:
            model = RTDETR(config['path'])

        nome_arquivo_video = config['nome'].replace(" + ", "_").replace("-", "") + ".mp4"
        caminho_video_saida = os.path.join(pasta_saida, nome_arquivo_video)

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_video = cv2.VideoWriter(caminho_video_saida, fourcc, fps_video, (largura_video, altura_video))

        results = model.track(source=video_teste, device='0', persist=True, tracker="custom_bytetrack.yaml", stream=True, verbose=False)

        for i, result in enumerate(results):
            frame_idx = primeiro_frame_gt + i
            frame_img = result.orig_img.copy()

            if result.boxes is not None and result.boxes.id is not None:
                boxes_xywh = result.boxes.xywh.cpu().numpy()
                ids = result.boxes.id.int().cpu().tolist()

                for box, track_id in zip(boxes_xywh, ids):
                    cx, cy, w, h = box
                    tl_x = cx - (w / 2)
                    tl_y = cy - (h / 2)

                    x1, y1 = int(tl_x), int(tl_y)
                    x2, y2 = int(tl_x + w), int(tl_y + h)

                    cv2.rectangle(frame_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.putText(frame_img, f"Pred:{track_id}", (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            gt_frame = df_gt[df_gt['frame'] == frame_idx]
            gt_ids = gt_frame['id'].tolist()
            gt_boxes = gt_frame[['bb_left', 'bb_top', 'bb_width', 'bb_height']].values.tolist()

            for gt_id, gt_box in zip(gt_ids, gt_boxes):
                l, t, bw, bh = gt_box
                x1, y1 = int(l), int(t)
                x2, y2 = int(l + bw), int(t + bh)

                cv2.rectangle(frame_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame_img, f"GT:{int(gt_id)}", (x1, y2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            cv2.putText(frame_img, "VERDE: Real (GT)", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame_img, "VERMELHO: Modelo", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            cv2.putText(frame_img, f"Frame: {frame_idx}", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

            out_video.write(frame_img)

        out_video.release()
        print(f"🎬 Sucesso! Vídeo salvo em: {caminho_video_saida}\n")

    except Exception as e:
        print(f"❌ Erro ao gerar vídeo para {config['nome']}: {e}\n")

    finally:
        if 'out_video' in locals():
            out_video.release()
        if 'model' in locals(): del model
        if 'results' in locals(): del results
        torch.cuda.empty_cache()
        gc.collect()

print("🎉 Todos os vídeos foram gerados!")

## Fase 4: Geração dos Gráficos com Plotly

In [ ]:
!pip install -U "kaleido==0.2.1"

In [ ]:
import pandas as pd
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Configurações de Diretório
# ==========================================
dir_graficos = "/content/drive/MyDrive/Lesc-Bee videos/graficos_artigo"
os.makedirs(dir_graficos, exist_ok=True)

csv_gpu = "/content/drive/MyDrive/Lesc-Bee videos/resultados_modelos_gpu.csv"
csv_cpu = "/content/drive/MyDrive/Lesc-Bee videos/resultados_modelos_cpu.csv"
csv_tracking = "/content/drive/MyDrive/Lesc-Bee videos/resultados_tracking_gpu.csv"

print(f"📁 Os gráficos serão salvos em: {dir_graficos}\n")

# ==========================================
# 2. Carregamento de Dados
# ==========================================
try:
    df_gpu_carregado = pd.read_csv(csv_gpu)
    df_cpu_carregado = pd.read_csv(csv_cpu)
    df_det = pd.concat([df_gpu_carregado, df_cpu_carregado], ignore_index=True)
    print("✅ Dados de detecção (GPU e CPU) carregados com sucesso.")
except FileNotFoundError:
    print("⚠️ CSV de detecção não encontrado.")
    df_det = pd.DataFrame()

try:
    df_track = pd.read_csv(csv_tracking)
    print("✅ Dados de tracking carregados com sucesso.")
except FileNotFoundError:
    print("⚠️ CSV de tracking não encontrado.")
    df_track = pd.DataFrame()

if not df_det.empty:

    if 'Formato' in df_det.columns:
        df_det['Formato_Limpo'] = df_det['Formato'].str.replace('ONNX_', '').str.replace('OpenVINO_', '')
    df_det['Latencia_ms'] = np.where(df_det['Dispositivo'] == 'GPU', df_det['Latencia_ms_GPU'], df_det['Latencia_ms_CPU'])
    df_det['FPS'] = np.where(df_det['Dispositivo'] == 'GPU', df_det['FPS_GPU'], df_det['FPS_CPU'])

    # =====================================================================
    # GRÁFICO 2: Impacto da Quantização (SUBPLOTS - 3 em 1)
    # =====================================================================
    # Filtra pelos formatos limpados de interesse (FP32, FP16, INT8).
    # Caso haja dados de CPU e GPU para o mesmo modelo/formato, selecionamos os dispositivos padrão
    # (ex: GPU para FP32/FP16 e CPU para INT8) para não duplicar barras. Ajuste se necessário.
    df_grafico2 = df_det[
        ((df_det['Formato_Limpo'] == 'FP32') & (df_det['Dispositivo'] == 'GPU')) |
        ((df_det['Formato_Limpo'] == 'FP16') & (df_det['Dispositivo'] == 'GPU')) |
        ((df_det['Formato_Limpo'] == 'INT8') & (df_det['Dispositivo'] == 'CPU')) # ou GPU, dependendo de onde rodou o INT8
    ].copy()

    # Define a legenda apenas pelo formato, removendo o nome do dispositivo
    df_grafico2['Legenda'] = df_grafico2['Formato_Limpo']

    ordem_legenda = ['FP32', 'FP16', 'INT8']
    cores_legenda = {'FP32': '#636EFA', 'FP16': '#EF553B', 'INT8': '#00CC96'}

    grupos_graficos = [
        {
            'nome_arquivo': '2a_degradacao_map50_comparativo.png',
            'titulo_geral': 'Comparativo de mAP 50 por Classe (Quantização)',
            'metricas': ['mAP50_Global', 'mAP50_bee', 'mAP50_pollen']
        },
        {
            'nome_arquivo': '2b_degradacao_map5095_comparativo.png',
            'titulo_geral': 'Comparativo de mAP 50-95 por Classe (Quantização)',
            'metricas': ['mAP50-95_Global', 'mAP50-95_bee', 'mAP50-95_pollen']
        }
    ]

    for grupo in grupos_graficos:
        if not all(m in df_grafico2.columns for m in grupo['metricas']):
            continue

        fig2 = make_subplots(
            rows=1, cols=3,
            subplot_titles=("Global", "Abelhas", "Pólen"),
            shared_yaxes=True,
            horizontal_spacing=0.02
        )

        for i, metrica in enumerate(grupo['metricas']):
            coluna_atual = i + 1

            for legenda in ordem_legenda:
                df_form = df_grafico2[df_grafico2['Legenda'] == legenda]
                if not df_form.empty:
                    mostrar_legenda = True if coluna_atual == 1 else False

                    fig2.add_trace(go.Bar(
                        x=df_form['Modelo'],
                        y=df_form[metrica],
                        name=legenda,
                        marker_color=cores_legenda.get(legenda),
                        text=df_form[metrica].round(2),
                        textposition='auto',
                        textangle=0,
                        showlegend=mostrar_legenda
                    ), row=1, col=coluna_atual)

        fig2.update_layout(
            title_text=grupo['titulo_geral'],
            title_x=0.5,
            barmode='group',
            template='plotly_white',
            legend_title_text='Formato',
            yaxis=dict(range=[0, 1.1]),
            width=1200,
            height=600
        )

        caminho_fig2 = f"{dir_graficos}/{grupo['nome_arquivo']}"
        fig2.write_image(caminho_fig2)
        print(f"\n📸 Gráfico Comparativo ({grupo['titulo_geral']}) gerado com sucesso!")

    # =====================================================================
    # GRÁFICO 3: Desempenho Multiclasse (Radar) com mAP50-95
    # =====================================================================
    df_radar = df_det[(df_det['Formato'] == 'ONNX_FP32') & (df_det['Dispositivo'] == 'GPU')].copy()

    categorias_possiveis = [
        'Precision_bee', 'Recall_bee', 'mAP50_bee', 'mAP50-95_bee',
        'Precision_pollen', 'Recall_pollen', 'mAP50_pollen', 'mAP50-95_pollen'
    ]
    categorias = [c for c in categorias_possiveis if c in df_radar.columns]

    if categorias:
        fig3 = go.Figure()
        for index, row in df_radar.iterrows():
            valores = [row[cat] for cat in categorias]
            if len(valores) == len(categorias):
                valores.append(valores[0])
                cat_fechadas = categorias + [categorias[0]]
                fig3.add_trace(go.Scatterpolar(
                    r=valores, theta=[c.replace('_', ' ') for c in cat_fechadas], fill='toself', name=row['Modelo']
                ))

        fig3.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0.0, 1.0])),
            title='Desempenho Multiclasse (Abelha vs Pólen) - ONNX FP32 (GPU)',
            template='plotly_white'
        )
        fig3.write_image(f"{dir_graficos}/3_desempenho_radar_classes.png", width=800, height=700)
        print("\n📸 Gráfico 3 (Radar Multiclasse com mAP50-95) gerado.")

# =====================================================================
# RASTREAMENTO (Gráficos 4 e 5)
# =====================================================================
if not df_track.empty:
    fig4 = go.Figure()
    fig4.add_trace(go.Bar(x=df_track['Modelo'], y=df_track['MOTA (%)'], name='MOTA (%)', marker_color='indianred', text=df_track['MOTA (%)'], textposition='auto', textangle=0))
    fig4.add_trace(go.Bar(x=df_track['Modelo'], y=df_track['IDF1 (%)'], name='IDF1 (%)', marker_color='lightsalmon', text=df_track['IDF1 (%)'], textposition='auto', textangle=0))
    fig4.update_layout(title='Desempenho de Rastreamento (MOTA e IDF1)', xaxis_title='Detector + ByteTrack', yaxis_title='Pontuação (%)', barmode='group', template='plotly_white')
    fig4.write_image(f"{dir_graficos}/4_tracking_mota_idf1.png", width=900, height=600)
    print("\n📸 Gráfico 4 (Tracking) gerado.")

    fig5 = go.Figure()
    erros = ['Falsos Positivos (FP)', 'Falsos Negativos (FN)', 'ID Switches (IDSW)']
    cores_erros = ['#EF553B', '#FFA15A', '#AB63FA']
    for erro, cor in zip(erros, cores_erros):
        if erro in df_track.columns:
            fig5.add_trace(go.Bar(
                x=df_track['Modelo'], y=df_track[erro], name=erro, marker_color=cor,
                text=df_track[erro], textposition='auto', textangle=0
            ))
    fig5.update_layout(title='Composição de Erros de Rastreamento', xaxis_title='Modelo', yaxis_title='Quantidade Absoluta', barmode='stack', template='plotly_white')
    fig5.write_image(f"{dir_graficos}/5_tracking_erros_empilhados.png", width=900, height=600)
    print("📸 Gráfico 5 (Erros de Tracking) gerado.")

# =====================================================================
# TABELAS MATPLOTLIB (Agora puxando dados da GPU)
# =====================================================================
if not df_gpu_carregado.empty:
    print("\n📊 Gerando versões em tabela dos resultados da GPU (Matplotlib)...")

    configs_tabelas = [
        {"nome": "Global", "cols": ["Modelo", "Formato", "FPS_GPU", "Latencia_ms_GPU", "mAP50_Global", "mAP50-95_Global"], "sort": "mAP50_Global"},
        {"nome": "Abelhas", "cols": ["Modelo", "Formato", "FPS_GPU", "Precision_bee", "Recall_bee", "mAP50_bee", "mAP50-95_bee"], "sort": "mAP50_bee"},
        {"nome": "Polen", "cols": ["Modelo", "Formato", "FPS_GPU", "Precision_pollen", "Recall_pollen", "mAP50_pollen", "mAP50-95_pollen"], "sort": "mAP50_pollen"}
    ]

    for cfg in configs_tabelas:
        colunas_presentes = [c for c in cfg["cols"] if c in df_gpu_carregado.columns]
        if len(colunas_presentes) < 2:
            continue

        df_tabela = df_gpu_carregado[colunas_presentes].copy()
        df_tabela['Formato'] = df_tabela['Formato'].str.replace('ONNX_', '').str.replace('OpenVINO_', '')

        if cfg["sort"] in df_tabela.columns:
            df_tabela = df_tabela.sort_values(by=cfg["sort"], ascending=False)

        caminho_img = f"{dir_graficos}/tabela_resultados_gpu_{cfg['nome'].lower()}.png"

        largura = 11 + (1.5 if len(colunas_presentes) > 6 else 0)
        fig, ax = plt.subplots(figsize=(largura, 1 + 0.4 * len(df_tabela)))
        ax.axis('off')
        ax.axis('tight')

        tabela = ax.table(cellText=df_tabela.round(3).values, colLabels=df_tabela.columns, cellLoc='center', loc='center')
        tabela.auto_set_font_size(False)
        tabela.set_fontsize(10)
        tabela.scale(1.2, 1.5)

        for i, col in enumerate(df_tabela.columns):
            celula = tabela[0, i]
            celula.set_text_props(weight='bold', color='white')
            celula.set_facecolor('#4C72B0')

        plt.title(f"Resultados GPU - {cfg['nome']}", pad=20, fontweight='bold')
        plt.savefig(caminho_img, bbox_inches='tight', dpi=300)
        plt.close()
        print(f"🖼️ Tabela {cfg['nome']} salva em: {caminho_img}")

print("\n🎉 Processo de geração concluído!")

In [ ]:
import sys
import platform
import pkg_resources
import torch
import ultralytics
import pandas
import numpy
import cv2
import roboflow
import plotly
import onnx
import onnxruntime

print("--- Versões das Bibliotecas Principais ---")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Roboflow: {roboflow.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Numpy: {numpy.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"Plotly: {plotly.__version__}")
print(f"ONNX: {onnx.__version__}")
print(f"ONNX Runtime: {onnxruntime.__version__}")

try:
    import openvino
    print(f"OpenVINO: {openvino.__version__}")
except ImportError:
    print("OpenVINO: Não instalado ou não encontrado.")

print("\n--- Gerando requirements.txt ---")
# Gera um requirements.txt com todas as bibliotecas e suas versões exatas
!pip freeze > requirements.txt
print("✅ requirements.txt gerado com sucesso!")
print("Por favor, verifique o arquivo requirements.txt na pasta raiz do Colab.")

# Fase 5: Gerar Video com trackin

In [ ]:
MODEL_PATH = f'/content/Modelos/rtdetr/best.pt'
VIDEO_PATH = "/content/video_validacao_mota_30fps.mp4"
FPS = 30.0
HORA_INICIO_SEGUNDOS = 13 * 3600

Extrair o primerio frame do video

In [ ]:
# -*- coding: utf-8 -*-
"""
Script auxiliar para salvar o primeiro frame de um vídeo como uma imagem.

Como usar:
1.  Certifique-se de que o `VIDEO_PATH` está correto.
2.  Execute o script. Ele criará um arquivo de imagem chamado 'primeiro_frame.png'.
3.  Abra a imagem 'primeiro_frame.png' em seu computador com um editor de
    imagens (como Paint, GIMP, Photoshop, etc.).
4.  Mova o cursor do mouse sobre a imagem para ver as coordenadas (x, y) e
    anote os valores que você precisa para a 'AREA_ENTRADA' nos outros scripts.
"""
import cv2

# --- Configuração ---
OUTPUT_IMAGE_PATH = "primeiro_frame.png"
# --------------------

def salvar_primeiro_frame():
    """
    Função principal que carrega o vídeo e salva seu primeiro frame como imagem.
    """
    # 1) Carrega o vídeo
    print(f"Abrindo o vídeo: '{VIDEO_PATH}'...")
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print(f"Erro: Não foi possível abrir o vídeo em '{VIDEO_PATH}'")
        return

    # 2) Lê o primeiro frame
    print("Lendo o primeiro frame...")
    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("Erro: Não foi possível ler o primeiro frame do vídeo.")
        return

    # 3) Tenta salvar o frame como um arquivo de imagem
    try:
        print(f"Salvando o frame como '{OUTPUT_IMAGE_PATH}'...")
        cv2.imwrite(OUTPUT_IMAGE_PATH, frame)
        print("\n--- Sucesso! ---")
        print(f"O primeiro frame foi salvo como '{OUTPUT_IMAGE_PATH}'.")
        print("\nPróximos passos:")
        print(f"1. Abra o arquivo '{OUTPUT_IMAGE_PATH}' no seu computador.")
        print("2. Use um editor de imagens para encontrar as coordenadas (x,y) da entrada da colmeia.")
    except Exception as e:
        print(f"Erro ao salvar a imagem: {e}")

# Executa a função principal
if __name__ == "__main__":
    salvar_primeiro_frame()

Registrar a coordenada da entrada da colmeia.

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

# --- Defina a área da entrada da colmeia (x_inicio, y_inicio, largura, altura) ---
#    !!!! IMPORTANTE: Você precisará ajustar estes valores para o seu vídeo! !!!!
#    Use um editor de imagens como o Paint para encontrar as coordenadas aproximadas.

AREA_ENTRADA = (251, 300, 100, 100)

# --- Configuração ---
OUTPUT_IMAGE_PATH = "primeiro_frame.png"
# A AREA_ENTRADA já foi definida no script anterior

# --- Visualização ---
print(f"Carregando a imagem: '{OUTPUT_IMAGE_PATH}'")
image = cv2.imread(OUTPUT_IMAGE_PATH)

if image is None:
    print(f"Erro: Não foi possível carregar a imagem em '{OUTPUT_IMAGE_PATH}'")
else:
    # Desenha a área de entrada na imagem
    x_area, y_area, w_area, h_area = AREA_ENTRADA
    COR_AREA = (255, 255, 0) # Ciano em BGR
    cv2.rectangle(image, (x_area, y_area), (x_area + w_area, y_area + h_area), COR_AREA, 2, cv2.LINE_AA)

    print("Exibindo a imagem com a área marcada...")
    cv2_imshow(image)

In [ ]:
"""
Script para deteção e rastreamento de abelhas num vídeo utilizando YOLOv8.
Atualizado para usar o rastreador ByteTrack (ideal para insetos/câmaras estáticas)
e inclui um Vetor Preditivo para analisar o comportamento do Filtro de Kalman.
"""
import cv2
from ultralytics import YOLO
import datetime
import numpy as np
import csv
import math
from tqdm import tqdm

# ================================================
# 1. Configuração do Rastreador ByteTrack (Criar YAML)
# ================================================
BASE_CONF = 0.20
TRACK_HIGH_THRESH = BASE_CONF
TRACK_LOW_THRESH = BASE_CONF * 0.25
NEW_TRACK_THRESH = BASE_CONF * 1.25
CONF_THRESHOLD = TRACK_LOW_THRESH

yaml_content = f"""
tracker_type: bytetrack
track_high_thresh: 0.50
track_low_thresh: 0.10
new_track_thresh: 0.60
track_buffer: 360
match_thresh: 0.95
gmc_method: none
appearance_thresh: 0.25
with_reid: False
fuse_score: True
"""

tracker_config = "custom_bytetrack.yaml"
with open(tracker_config, "w") as f:
    f.write(yaml_content)

print(f"Ficheiro {tracker_config} criado com sucesso!")

# ================================================
# 2. Configuração Geral de Saída
# ================================================
OUTPUT_VIDEO_PATH = "video_rastreado_bytetrack_preditivo.mp4"
LOG_PATH = "eventos_abelhas_classificado.log"
TRAJECTORY_LOG_PATH = "trajetorias_abelhas.csv"

MAX_TRAJECTORY_POINTS_DISPLAY = 9999
MAX_FRAMES_AUSENTE = 360 # Tempo de vida da abelha no buffer
COR_AREA = (255, 255, 0) # Ciano
MIN_SPREAD_PIXELS = 15 # Filtro híbrido de ruídos

# ================================================
# 3. Funções Auxiliares
# ================================================
def format_timestamp(frame_idx, fps, start_time_seconds):
    video_time_seconds = frame_idx / fps
    total_seconds = start_time_seconds + video_time_seconds
    return str(datetime.timedelta(seconds=int(total_seconds)))

def draw_info_panel(frame, counters):
    panel_height = 150
    panel_color = (10, 10, 10)
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], panel_height), panel_color, -1)
    alpha = 0.6
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    font_color = (255, 255, 255)
    thickness = 2

    cv2.putText(frame, f"Entradas: {counters['entradas']}", (20, 30), font, font_scale, font_color, thickness)
    cv2.putText(frame, f"Saidas: {counters['saidas']}", (20, 60), font, font_scale, font_color, thickness)
    cv2.putText(frame, f"Indefinido: {counters['indefinido']}", (20, 90), font, font_scale, font_color, thickness)
    cv2.putText(frame, f"Ruidos Ignorados: {counters['falsos_estaticos']}", (20, 120), font, font_scale, (100, 100, 255), thickness)

    return frame

def calculate_intersection_area(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0
    return (x_right - x_left) * (y_bottom - y_top)

def save_trajectories_to_csv(data, filepath):
    header = ['ID_Abelha', 'Frame', 'Horario', 'Centro_X', 'Centro_Y', 'Classe', 'Confianca', 'Area_Pixels', 'Interseccao_Anterior', 'Tempo_Inferencia_ms', 'Distancia_Acumulada', 'Espalhamento_Maximo']
    try:
        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(data)
        print(f"Log de trajetórias guardado com sucesso em: {filepath}")
    except IOError as e:
        print(f"Erro ao guardar o ficheiro CSV de trajetórias: {e}")

def desenhar_vetor_preditivo(frame, trajetoria):
    """
    Desenha uma seta (Vetor de Inércia) indicando a previsão linear do rastreador.
    Se a abelha fizer uma curva brusca e a seta apontar para outro lado,
    é aqui que o Filtro de Kalman do rastreador falha e perde o ID.
    """
    # CORREÇÃO: O uso de min() garante que nunca vamos além do limite da lista.
    # Ex: se a lista tiver tamanho 5, o passo_atras máximo será 4.
    passo_atras = min(5, len(trajetoria) - 1)

    if passo_atras >= 1:
        p_atual = trajetoria[-1]
        p_anterior = trajetoria[-(passo_atras + 1)]

        # Calcula a velocidade direcional (dx, dy)
        dx = p_atual[0] - p_anterior[0]
        dy = p_atual[1] - p_anterior[1]

        # Fator de projeção (aumenta o tamanho da seta para melhor visualização)
        fator_projecao = 3
        p_futuro = (int(p_atual[0] + (dx * fator_projecao)),
                    int(p_atual[1] + (dy * fator_projecao)))

        # Desenha a seta apontando para o futuro
        cv2.arrowedLine(frame, p_atual, p_futuro, (0, 0, 255), 2, tipLength=0.3)

# ================================================
# 4. Início do Script Principal
# ================================================
print("A carregar o modelo YOLO...")
try:
    model = YOLO(MODEL_PATH)
except Exception as e:
    print(f"Aviso: Não foi possível carregar o modelo em {MODEL_PATH}. Verifica o caminho. Erro: {e}")
    # Descomenta a linha abaixo para testar com um modelo padrão se falhar
    # model = YOLO("yolov8n.pt")

print(f"A abrir o vídeo de entrada: {VIDEO_PATH}")
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = 0

if not cap.isOpened():
    print(f"Aviso: Não foi possível abrir o vídeo: {VIDEO_PATH}. Cria um ficheiro de teste ou corrige o caminho.")
    width, height = 640, 480
else:
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, FPS, (width, height))

log_file = open(LOG_PATH, "w", encoding="utf-8")
log_file.write("ID | Horário | Evento | Detalhes\n")
log_file.write("---|---|---|---\n")

# Dicionários e Variáveis de Controlo
trajetorias_ativas = {}
ultima_vez_visto = {}
ultima_bbox = {}
distancia_acumulada = {}
limites_trajetoria = {}

dados_trajetoria_csv = []
counters = {'entradas': 0, 'saidas': 0, 'indefinido': 0, 'ignorados_na_porta': 0, 'falsos_estaticos': 0}

print("A iniciar o rastreamento (ByteTrack) e classificação por trajetória...")
# A usar o novo tracker_config gerado no passo 1
results_stream = model.track(source=VIDEO_PATH, show=False, stream=True, conf=CONF_THRESHOLD, tracker=tracker_config, verbose=False)

frame_idx = 0

with tqdm(total=total_frames, desc="A analisar Vídeo", unit="frame") as pbar:
    try:
        for result in results_stream:
            frame_idx += 1
            horario = format_timestamp(frame_idx, FPS, HORA_INICIO_SEGUNDOS)
            frame_anotado = result.plot()

            infer_time_ms = result.speed.get('inference', 0.0)

            x_area, y_area, w_area, h_area = AREA_ENTRADA
            cv2.rectangle(frame_anotado, (x_area, y_area), (x_area + w_area, y_area + h_area), COR_AREA, 2, cv2.LINE_AA)

            vistos_neste_frame = set()
            if result.boxes.id is not None:
                ids = result.boxes.id.cpu().numpy().astype(int)
                boxes_xyxy = result.boxes.xyxy.cpu().numpy()
                clses = result.boxes.cls.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()

                for i, track_id in enumerate(ids):
                    vistos_neste_frame.add(track_id)
                    x1, y1, x2, y2 = boxes_xyxy[i]
                    cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)

                    confianca = float(confs[i])
                    area_pixel = (x2 - x1) * (y2 - y1)

                    bbox_atual = [x1, y1, x2, y2]
                    interseccao = 0.0
                    if track_id in ultima_bbox:
                        interseccao = calculate_intersection_area(ultima_bbox[track_id], bbox_atual)
                    ultima_bbox[track_id] = bbox_atual

                    # INICIALIZAÇÃO DE UMA NOVA TRAJETÓRIA
                    if track_id not in trajetorias_ativas:
                        trajetorias_ativas[track_id] = []
                        distancia_acumulada[track_id] = 0.0
                        limites_trajetoria[track_id] = [cx, cx, cy, cy]
                        log_file.write(f"{track_id} | {horario} | DETETADO | Nova abelha em cena\n")

                    # CÁLCULOS DO FILTRO DE MOVIMENTO
                    if len(trajetorias_ativas[track_id]) > 0:
                        last_x, last_y = trajetorias_ativas[track_id][-1]
                        dist = math.hypot(cx - last_x, cy - last_y)
                        distancia_acumulada[track_id] += dist

                    limites_trajetoria[track_id][0] = min(limites_trajetoria[track_id][0], cx)
                    limites_trajetoria[track_id][1] = max(limites_trajetoria[track_id][1], cx)
                    limites_trajetoria[track_id][2] = min(limites_trajetoria[track_id][2], cy)
                    limites_trajetoria[track_id][3] = max(limites_trajetoria[track_id][3], cy)

                    spread_x = limites_trajetoria[track_id][1] - limites_trajetoria[track_id][0]
                    spread_y = limites_trajetoria[track_id][3] - limites_trajetoria[track_id][2]
                    espalhamento_maximo = max(spread_x, spread_y)

                    trajetorias_ativas[track_id].append((cx, cy))
                    ultima_vez_visto[track_id] = frame_idx

                    classe_nome = result.names[clses[i]]
                    dados_trajetoria_csv.append([track_id, frame_idx, horario, cx, cy, classe_nome, confianca, area_pixel, interseccao, infer_time_ms, distancia_acumulada[track_id], espalhamento_maximo])

            # Lógica de Classificação (Remoção do Buffer)
            ids_para_remover = []
            for track_id in trajetorias_ativas.keys():
                if frame_idx - ultima_vez_visto.get(track_id, frame_idx) > MAX_FRAMES_AUSENTE:
                    ids_para_remover.append(track_id)

            for track_id in ids_para_remover:
                trajetoria = trajetorias_ativas[track_id]

                if len(trajetoria) >= 2:
                    ponto_inicial = trajetoria[0]
                    ponto_final = trajetoria[-1]

                    p_ini_na_area = (x_area <= ponto_inicial[0] <= x_area + w_area) and (y_area <= ponto_inicial[1] <= y_area + h_area)
                    p_fim_na_area = (x_area <= ponto_final[0] <= x_area + w_area) and (y_area <= ponto_final[1] <= y_area + h_area)

                    spread_x = limites_trajetoria[track_id][1] - limites_trajetoria[track_id][0]
                    spread_y = limites_trajetoria[track_id][3] - limites_trajetoria[track_id][2]
                    espalhamento_final = max(spread_x, spread_y)

                    classificacao = "INDEFINIDO"

                    if espalhamento_final < MIN_SPREAD_PIXELS:
                        counters['falsos_estaticos'] += 1
                        classificacao = "FALSO_ESTATICO"
                    elif p_fim_na_area and not p_ini_na_area:
                        counters['entradas'] += 1
                        classificacao = "ENTRADA"
                    elif p_ini_na_area and not p_fim_na_area:
                        counters['saidas'] += 1
                        classificacao = "SAIDA"
                    elif p_ini_na_area and p_fim_na_area:
                        counters['ignorados_na_porta'] += 1
                        classificacao = "IGNORADO_NA_PORTA"
                    else:
                        counters['indefinido'] += 1

                    log_file.write(f"{track_id} | {horario} | TRAJETORIA_FIM | Classificado como: {classificacao} (Espalhamento: {espalhamento_final}px)\n")

                # Limpeza da memória
                del trajetorias_ativas[track_id]
                if track_id in ultima_vez_visto: del ultima_vez_visto[track_id]
                if track_id in ultima_bbox: del ultima_bbox[track_id]
                if track_id in distancia_acumulada: del distancia_acumulada[track_id]
                if track_id in limites_trajetoria: del limites_trajetoria[track_id]

            # Desenho das Trajetórias e Vetor Preditivo
            for track_id, pontos in trajetorias_ativas.items():
                if len(pontos) > 1:
                    # Desenha a linha verde (caminho percorrido)
                    pontos_para_desenho = pontos[-MAX_TRAJECTORY_POINTS_DISPLAY:]
                    pontos_np = np.array(pontos_para_desenho, dtype=np.int32).reshape((-1, 1, 2))
                    cv2.polylines(frame_anotado, [pontos_np], isClosed=False, color=(0, 255, 0), thickness=2)
                    cv2.circle(frame_anotado, pontos[-1], 4, (0, 0, 255), -1)

                    # Desenha a seta vermelha (previsão do rastreador)
                    desenhar_vetor_preditivo(frame_anotado, pontos)

            frame_com_info = draw_info_panel(frame_anotado, counters)
            writer.write(frame_com_info)

            pbar.set_postfix(In=counters['entradas'], Out=counters['saidas'], Ign=counters['ignorados_na_porta'], Ruidos=counters['falsos_estaticos'])
            pbar.update(1)

    except Exception as e:
        print(f"\nErro durante o processamento: {e}")

    finally:
        print("\nProcessamento concluído.")
        print("A classificar abelhas remanescentes no fim do vídeo...")
        for track_id, trajetoria in trajetorias_ativas.items():
            if len(trajetoria) >= 2:
                ponto_inicial = trajetoria[0]
                ponto_final = trajetoria[-1]
                p_ini_na_area = (x_area <= ponto_inicial[0] <= x_area + w_area) and (y_area <= ponto_inicial[1] <= y_area + h_area)
                p_fim_na_area = (x_area <= ponto_final[0] <= x_area + w_area) and (y_area <= ponto_final[1] <= y_area + h_area)

                spread_x = limites_trajetoria[track_id][1] - limites_trajetoria[track_id][0]
                spread_y = limites_trajetoria[track_id][3] - limites_trajetoria[track_id][2]
                espalhamento_final = max(spread_x, spread_y)

                if espalhamento_final < MIN_SPREAD_PIXELS:
                    counters['falsos_estaticos'] += 1
                elif p_fim_na_area and not p_ini_na_area:
                    counters['entradas'] += 1
                elif p_ini_na_area and not p_fim_na_area:
                    counters['saidas'] += 1
                elif p_ini_na_area and p_fim_na_area:
                    counters['ignorados_na_porta'] += 1
                else:
                    counters['indefinido'] += 1

        writer.release()
        save_trajectories_to_csv(dados_trajetoria_csv, TRAJECTORY_LOG_PATH)

        resumo = (
            f"\n\n--- RELATÓRIO FINAL ---\n"
            f"Total de Entradas Classificadas: {counters['entradas']}\n"
            f"Total de Saídas Classificadas: {counters['saidas']}\n"
            f"Total de Trajetórias Indefinidas: {counters['indefinido']}\n"
            f"Total de Ignoradas (Ficaram na Porta): {counters['ignorados_na_porta']}\n"
            f"Total de Falsos Estáticos (Ruídos Barrados): {counters['falsos_estaticos']}\n"
            f"---------------------------\n"
        )
        print(resumo)
        try:
            log_file.write(resumo)
            log_file.close()
        except:
            pass

        print(f"Vídeo anotado guardado em: {OUTPUT_VIDEO_PATH}")
        print(f"Log de eventos guardado em: {LOG_PATH}")